In [ ]:
import requests
import lxml.html
from pypdf import PdfReader
import pandas as pd
from google import genai
from pathlib import Path
from google.genai import types
import os

from dotenv import load_dotenv, find_dotenv
from pydantic import BaseModel, Field
from typing import List, Optional

from ingest_sc_cases import case_df
from ingest_sc_judges import judge_pd

In [25]:
# Cases with no opinion documents
case_df[~case_df["opinion_link"].str.contains("statecourt", na=False)]

,docket_no,title,state,date,type,pending,opinion_link
11,2026-00384,Williams v. Board of Elections of the State of...,New York,"March 19, 2026",Voting Rights and Elections,False,None
35,20220896,State v. Andrus,Utah,"August 7, 2025",Criminal Law,False,None
47,S-1-SC-40715,Franklin v. Martinez,New Mexico,"January 26, 2026",Criminal Law,False,None
88,22 OC 00022 1 B,Persaud-Zamora v. Cegavske,Nevada,"April 26, 2022",Voting Rights and Elections,False,None
101,EF2025-2536,Texas v. Bruck,Texas,"October 31, 2025",Reproductive Rights,False,None
103,2025AP002121,Voces de la Frontera v. Gerber,Wisconsin,"September 18, 2025",Government Structure,False,None
120,PR-123179,McVay v. Cockroft,Oklahoma,"July 28, 2025",Voting Rights and Elections,False,None
152,15-25-00093-CV,State v. City of San Antonio,Texas,"June 19, 2025",Reproductive Rights,False,None
207,CV01-23-14744,Adkins v. State,Idaho,"April 11, 2025",Reproductive Rights,False,None
228,SJC-13666,Gotay v. Creen,Massachusetts,"March 21, 2025",Civil Due Process,False,None


In [33]:
class Opinion(BaseModel):
    judge_name: str = Field(description="The full name of the judge giving an opinion.")
    opinion_summary: str = Field(
        description="An extremely brief description of the judge's opinion on the case."
    )


class Case(BaseModel):
    title: str = Field(description="The title of the case.")
    decision: str = Field(description="The final decision that was made for the case.")
    issue_debate: str = Field(description="The main issue being debated in the case.")
    judge_opinions: List[Opinion]

In [34]:
def read_opinion(pdf_link: str, model_id: str, client: genai.Client, prompt: str):
    resp = requests.get(pdf_link).content

    genai_resp = client.models.generate_content(
        model=model_id,
        contents=[types.Part.from_bytes(data=resp, mime_type="application/pdf"), prompt],
        config={
            "response_mime_type": "application/json",
            "response_json_schema": Case.model_json_schema(),
        },
    )

    return Case.model_validate_json(genai_resp.text)

In [ ]:
def analyze_state_cases(
    case_df: pd.DataFrame, judge_df: pd.DataFrame, prompt_start: str, client_info: dict
):
    prompt = prompt_start + "$DATAFRAME$" + judge_df.to_markdown(index=False)
    num_cases = len(case_df)
    model_id = client_info["model_id"]
    client = client_info["client"]

    file_dic = {}
    for i in range(num_cases):
        print(f"Querying case {i}")
        docket_no = case_df.iloc[i]["docket_no"].replace(" ", "-")
        state = case_df.iloc[i]["state"]
        date = case_df.iloc[i]["date"].replace(" ", "-")
        pdf_link = case_df.iloc[i]["opinion_link"]

        case_ref = "_".join([docket_no, state, date])

        opinion_resp = read_opinion(pdf_link, model_id, client, prompt)

        file_dic[case_ref] = {}
        file_dic[case_ref]["pdf_link"] = pdf_link
        file_dic[case_ref]["response"] = opinion_resp

    return file_dic

In [39]:
with open("prompt.txt", "r") as prompt_file:
    prompt_start = prompt_file.read()

load_dotenv(find_dotenv())

gemini_key = os.getenv("GEMINI_API_KEY")
client_info = {"model_id": "gemini-2.5-flash-lite", "client": genai.Client(api_key=gemini_key)}

lou_cases = case_df[case_df["state"] == "Louisiana"].head(5)
lou_judges = judge_pd[judge_pd["state"] == "LA"]

file_dic = analyze_state_cases(lou_cases, lou_judges, prompt_start, client_info)

Querying case 0
Querying case 1
Querying case 2
Querying case 3
Querying case 4


In [40]:
for key in file_dic.keys():
    print(f"Individual opinions for {key}")
    print(file_dic[key]["response"])

Individual opinions for 2025-O-00879_Louisiana_October-24,-2025
title='IN RE: JUDGE JENNIFER M. MEDLEY' decision='Judge Jennifer M. Medley is suspended for thirty days without pay and must reimburse the Commission $2,747.41 in costs.' issue_debate="The issue is whether Judge Medley's campaign activities, including false statements and unreported expenditures, violated judicial conduct rules and state law, with the defense arguing First Amendment protection for political speech versus the prosecution arguing for sanctions due to knowing falsehoods and misconduct." judge_opinions=[Opinion(judge_name='Cade Cole', opinion_summary='Not available'), Opinion(judge_name='Jay B. McCallum', opinion_summary='Not available'), Opinion(judge_name='Jefferson D. Hughes III', opinion_summary='Not available'), Opinion(judge_name='John Guidry', opinion_summary='Not available'), Opinion(judge_name='John L. Weimer', opinion_summary='Agrees with the majority findings related to dismissing Count I but findin

In [44]:
opinion_table = {"opinion_id": [], "name": [], "opinion": []}

for key in file_dic.keys():
    opinions = file_dic[key]["response"].model_dump()
    num_opinions = len(opinions["judge_opinions"])

    for i in range(num_opinions):
        opinion_table["opinion_id"].append(key)
        opinion_table["name"].append(opinions["judge_opinions"][i]["judge_name"])
        opinion_table["opinion"].append(opinions["judge_opinions"][i]["opinion_summary"])

opinion_pd = pd.DataFrame(opinion_table)
opinion_pd

,opinion_id,name,opinion
0,"2025-O-00879_Louisiana_October-24,-2025",Cade Cole,Not available
1,"2025-O-00879_Louisiana_October-24,-2025",Jay B. McCallum,Not available
2,"2025-O-00879_Louisiana_October-24,-2025",Jefferson D. Hughes III,Not available
3,"2025-O-00879_Louisiana_October-24,-2025",John Guidry,Not available
4,"2025-O-00879_Louisiana_October-24,-2025",John L. Weimer,Agrees with the majority findings related to d...
5,"2025-O-00879_Louisiana_October-24,-2025",Piper D. Griffin,Not available
6,"2024-KK-00737_Louisiana_May-9,-2025",Piper D. Griffin,Granted writ to determine if prosecutors may j...
7,"2024-KK-00737_Louisiana_May-9,-2025",Jefferson D. Hughes III,"Concurs, stating death penalty cases have a di..."
8,"2024-KK-00737_Louisiana_May-9,-2025",Jay B. McCallum,"Dissents, arguing that the majority's conclusi..."
9,"2024-KK-00737_Louisiana_May-9,-2025",Cade Cole,"Concurs in the result, agreeing that capital a..."
